# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Charger les données

In [ ]:
import json
import ast
from pathlib import Path

def load_pyomo_data(input_path="../data/toysarus_data.json"):
    """Charge les donnees externes depuis un fichier JSON."""
    input_file = Path(input_path)

    with open(input_file, "r") as f:
        data = json.load(f)

    def _convert_key(key):
        if not isinstance(key, str):
            return key
        if key.startswith("(") and key.endswith(")"):
            try:
                return ast.literal_eval(key)
            except Exception:
                return key
        try:
            return int(key)
        except Exception:
            return key

    # Convertir les dictionnaires de parametres indexes
    params = data.get("params", {})
    for pname, pval in list(params.items()):
        if isinstance(pval, dict):
            params[pname] = {_convert_key(k): v for k, v in pval.items()}

    cartesian = data.get("cartesian_data", {})
    for cname, cval in list(cartesian.items()):
        if isinstance(cval, dict):
            cartesian[cname] = {_convert_key(k): v for k, v in cval.items()}

    data["params"] = params
    data["cartesian_data"] = cartesian
    return data

# Charger les données
data = load_pyomo_data()

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.JOUETS = Set(initialize=data['sets']['JOUETS'])
model.USINES = Set(initialize=data['sets']['USINES'])
model.ARC = Set(dimen=2, initialize=[(i0,i1) for i0 in model.USINES for i1 in model.JOUETS])

## 🔹 Parameters

In [ ]:
model.revenu = Param(model.JOUETS, initialize=data['params']['revenu'], within=NonNegativeReals)
model.cout_dem = Param(model.JOUETS, initialize=data['params']['cout_dem'], within=NonNegativeReals)
model.prod_max = Param(model.USINES, initialize=data['params']['prod_max'], within=NonNegativeReals)
model.ponderation = Param(model.USINES, model.JOUETS, initialize=data['cartesian_data']['ponderation'], within=NonNegativeReals)
model.bigM = Param(initialize=data['params']['bigM'], within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.x = Var(model.JOUETS, domain=NonNegativeReals)
model.y = Var(model.JOUETS, domain=Binary)
model.z = Var(model.USINES, domain=Binary)

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=sum(model.z[u] for u in model.USINES) == 1)
model.c_for_0 = ConstraintList()
for u in model.USINES:
    model.c_for_0.add(sum(model.ponderation[u, j] * model.x[j] for j in model.JOUETS) <= model.prod_max[u] + model.bigM * ( 1 - model.z[u] ))
model.c_for_1 = ConstraintList()
for j in model.JOUETS:
    model.c_for_1.add(model.x[j] <= model.bigM * model.y[j])
# @BIN/@GIN directive already handled in variable declarations

## 🔹 O

In [ ]:
# @BIN/@GIN directive already handled in variable declarations

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.revenu[j] * model.x[j] - model.cout_dem[j] * model.y[j] for j in model.JOUETS), sense=maximize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')